# 전투 적합 여부·회복기간 예측
시작: 2026-10-09 14:45 KST · 종료: 16:35 KST
문제·채점 기준은 README.md를 확인하세요. 아래 기본 예제를 개선하고 submission.csv를 저장하세요.
서버 전체에서 커널 하나만 사용합니다. 완료 후 Kernel → Shutdown으로 실행 자리를 반납하세요.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
DATA=Path('data')
train=pd.read_csv(DATA/'train.csv'); test=pd.read_csv(DATA/'test.csv'); sample=pd.read_csv(DATA/'sample_submission.csv')
print(train.shape, test.shape)


In [ ]:
features=[c for c in test.columns if c!='soldier_id']
X=train[features]
a,b=train_test_split(np.arange(len(train)),test_size=.25,random_state=42,stratify=train.status)
classifier=RandomForestClassifier(n_estimators=64,n_jobs=2,random_state=42)
regressor=RandomForestRegressor(n_estimators=64,n_jobs=2,random_state=42)
classifier.fit(X.iloc[a],train.status.iloc[a]);regressor.fit(X.iloc[a],train.recovery_days.iloc[a])
print('Validation accuracy:',accuracy_score(train.status.iloc[b],classifier.predict(X.iloc[b])))
print('Validation recovery RMSE:',mean_squared_error(train.recovery_days.iloc[b],regressor.predict(X.iloc[b]))**.5)
# TODO: 분류·회귀 특징과 검증 전략을 개선하세요.
classifier.fit(X,train.status);regressor.fit(X,train.recovery_days)
submission=pd.DataFrame({'soldier_id':test.soldier_id,'status':classifier.predict(test[features]),'recovery_days':np.maximum(0,regressor.predict(test[features]))})[sample.columns]
submission.to_csv('submission.csv',index=False)


In [ ]:
check = pd.read_csv('submission.csv')
assert check.columns.tolist() == sample.columns.tolist()
assert check.iloc[:,0].tolist() == sample.iloc[:,0].tolist()
assert check.iloc[:,0].is_unique and not check.isna().any().any()
print('제출 형식 확인:', check.shape)
